# MNIST Digit Recognition — CNN Training

This notebook trains a Convolutional Neural Network (CNN) on the MNIST handwritten digit dataset,
augmented with a synthetic **Junk** class (Class 10) to reject random scribbles and non-digit drawings.

**Goal**: Build a model that classifies handwritten digits (0-9) with >99% accuracy *and*
confidently rejects non-digit inputs, then export it for real-time inference in our web app.

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from PIL import Image, ImageDraw, ImageFilter
import os
import random
import time

# Set style for plots
plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Hyperparameters

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 10
NUM_CLASSES = 11         # 10 digits + 1 junk class

# Junk dataset sizes
NUM_JUNK_TRAIN = 6000    # ~10% of MNIST training set
NUM_JUNK_TEST = 1000     # ~10% of MNIST test set

print(f'Batch Size: {BATCH_SIZE}')
print(f'Learning Rate: {LEARNING_RATE}')
print(f'Epochs: {NUM_EPOCHS}')
print(f'Classes: {NUM_CLASSES} (0-9 digits + junk)')

## 3. Synthetic Junk Dataset

To teach the model to reject random scribbles, we synthesize images of
random lines, shapes, arcs, and blurred patterns. These are labelled as
Class 10 ("Junk").

In [ ]:
class JunkDataset(Dataset):
    """Generates synthetic 28x28 images of random scribbles for the 'junk' class."""

    def __init__(self, size, transform=None):
        self.size = size
        self.transform = transform

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        img = Image.new('L', (28, 28), color=0)
        draw = ImageDraw.Draw(img)

        num_shapes = random.randint(1, 5)
        for _ in range(num_shapes):
            shape_type = random.choice(['line', 'ellipse', 'rectangle', 'arc'])
            x1 = random.randint(0, 20)
            y1 = random.randint(0, 20)
            x2 = random.randint(x1 + 2, 28)
            y2 = random.randint(y1 + 2, 28)
            width = random.randint(1, 4)
            color = random.randint(100, 255)

            if shape_type == 'line':
                draw.line([(x1, y1), (x2, y2)], fill=color, width=width)
            elif shape_type == 'ellipse':
                draw.ellipse([x1, y1, x2, y2], outline=color, width=width)
            elif shape_type == 'rectangle':
                draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
            elif shape_type == 'arc':
                draw.arc([x1, y1, x2, y2], start=0, end=random.randint(90, 360),
                         fill=color, width=width)

        if random.random() > 0.5:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.2)))

        if self.transform:
            img = self.transform(img)

        return img, 10  # Label 10 = Junk


print('JunkDataset class defined.')

## 4. Load & Combine Datasets

In [ ]:
# Data transforms — normalize to mean=0.1307, std=0.3081 (MNIST stats)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Standard MNIST
mnist_train = datasets.MNIST(
    root='../data', train=True, download=True, transform=transform
)
mnist_test = datasets.MNIST(
    root='../data', train=False, download=True, transform=transform
)

# Synthetic junk data
junk_train = JunkDataset(NUM_JUNK_TRAIN, transform=transform)
junk_test = JunkDataset(NUM_JUNK_TEST, transform=transform)

# Combined datasets
train_dataset = ConcatDataset([mnist_train, junk_train])
test_dataset = ConcatDataset([mnist_test, junk_test])

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Training samples: {len(train_dataset)} (MNIST: {len(mnist_train)}, Junk: {len(junk_train)})')
print(f'Test samples:     {len(test_dataset)} (MNIST: {len(mnist_test)}, Junk: {len(junk_test)})')
print(f'Image shape:      {mnist_train[0][0].shape}')

## 5. Visualize Sample Data

A few MNIST digits alongside some synthetic junk images.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Sample MNIST Digits', fontsize=16, color='white')

for i, ax in enumerate(axes.flat):
    image, label = mnist_train[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=10, color='#00ff88')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Sample Junk Images (Class 10)', fontsize=16, color='white')

for i, ax in enumerate(axes.flat):
    image, label = junk_train[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=10, color='#ff6090')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. CNN Model Architecture

```
Input (1x28x28)
  │
  ├─► Conv2D(1→32, 3x3) → BatchNorm → ReLU → MaxPool(2x2)  →  [32x14x14]
  │
  ├─► Conv2D(32→64, 3x3) → BatchNorm → ReLU → MaxPool(2x2) →  [64x7x7]
  │
  ├─► Flatten → [3136]
  │
  ├─► Linear(3136→128) → ReLU → Dropout(0.5)
  │
  └─► Linear(128→11) → Output probabilities (0-9 digits + junk)
```

In [ ]:
class MNISTNet(nn.Module):
    """CNN for MNIST digit classification + junk rejection."""

    def __init__(self, num_classes=NUM_CLASSES):
        super(MNISTNet, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Pooling & dropout
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout1 = nn.Dropout2d(0.25)
        self.dropout2 = nn.Dropout(0.5)

        # Fully connected layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Conv block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Conv block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout1(x)

        # Flatten
        x = x.view(-1, 64 * 7 * 7)

        # FC layers
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)

        return x


model = MNISTNet().to(device)
print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

## 7. Training Setup

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f'Loss: CrossEntropyLoss')
print(f'Optimizer: Adam (lr={LEARNING_RATE})')
print(f'Scheduler: StepLR (step=5, gamma=0.5)')

## 8. Training Loop

In [ ]:
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # ─── Training ───
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # ─── Evaluation ───
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_loss /= len(test_loader)
    test_acc = 100. * correct / total
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    scheduler.step()

    elapsed = time.time() - start
    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}] ({elapsed:.1f}s)  '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  │  '
          f'Test Loss: {test_loss:.4f}  Test Acc: {test_acc:.2f}%')

print(f'\n✅ Training complete! Best test accuracy: {max(test_accuracies):.2f}%')

## 9. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, NUM_EPOCHS + 1)

# Loss curve
ax1.plot(epochs_range, train_losses, 'o-', color='#00ff88', label='Train Loss', linewidth=2)
ax1.plot(epochs_range, test_losses, 's-', color='#ff6090', label='Test Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss Curve', fontsize=14, color='white')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Accuracy curve
ax2.plot(epochs_range, train_accuracies, 'o-', color='#00e5ff', label='Train Accuracy', linewidth=2)
ax2.plot(epochs_range, test_accuracies, 's-', color='#ffd740', label='Test Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Accuracy Curve', fontsize=14, color='white')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Confusion Matrix

In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

class_labels = [str(i) for i in range(10)] + ['Junk']

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', ax=ax,
            xticklabels=class_labels, yticklabels=class_labels,
            linewidths=0.5, linecolor='gray')
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title('Confusion Matrix on Test Set (11 classes)', fontsize=15)
plt.tight_layout()
plt.show()

# Per-class accuracy
print('\nPer-class Accuracy:')
print('-' * 30)
for i in range(NUM_CLASSES):
    class_acc = cm[i, i] / cm[i].sum() * 100
    label = f'Digit {i}' if i < 10 else 'Junk   '
    print(f'  {label}: {class_acc:.1f}%')

## 11. Sample Predictions

In [ ]:
model.eval()
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

indices = np.random.choice(len(test_dataset), 10, replace=False)

for i, ax in enumerate(axes.flat):
    image, true_label = test_dataset[indices[i]]

    with torch.no_grad():
        output = model(image.unsqueeze(0).to(device))
        probs = F.softmax(output, dim=1).cpu().numpy()[0]
        pred_label = probs.argmax()
        confidence = probs[pred_label] * 100

    ax.imshow(image.squeeze(), cmap='gray')
    color = '#00ff88' if pred_label == true_label else '#ff4444'
    pred_str = str(pred_label) if pred_label < 10 else 'Junk'
    true_str = str(true_label) if true_label < 10 else 'Junk'
    ax.set_title(f'Pred: {pred_str} ({confidence:.1f}%)\nTrue: {true_str}',
                 fontsize=10, color=color)
    ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)',
             fontsize=14, color='white')
plt.tight_layout()
plt.show()

## 12. Save Model

In [ ]:
os.makedirs('../model', exist_ok=True)

model_path = '../model/mnist_cnn.pth'
torch.save(model.state_dict(), model_path)

file_size = os.path.getsize(model_path) / 1024
print(f'✅ Model saved to: {model_path}')
print(f'   File size: {file_size:.1f} KB')

# Quick verification — reload and test
test_model = MNISTNet().to(device)
test_model.load_state_dict(torch.load(model_path, map_location=device))
test_model.eval()

with torch.no_grad():
    sample, label = test_dataset[0]
    output = test_model(sample.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).item()
    print(f'\n   Verification — True: {label}, Predicted: {pred} ✓' if pred == label
          else f'\n   Verification — True: {label}, Predicted: {pred} ✗')

## 13. Export to ONNX

Convert the trained model to ONNX format for lightweight deployment with ONNX Runtime.

In [ ]:
model.eval()

dummy_input = torch.randn(1, 1, 28, 28).to(device)
onnx_path = '../model/mnist_cnn.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}},
    opset_version=17
)

print(f'✅ Exported to: {onnx_path}')
print(f'   File size: {os.path.getsize(onnx_path) / 1024:.1f} KB')

---

### ✅ Next Steps

The trained model (`model/mnist_cnn.pth`) and its ONNX export (`model/mnist_cnn.onnx`) are ready.

Run the web app:
```bash
python app.py
```
Then open `http://localhost:5000` to draw digits and see real-time predictions!

The model now recognizes 0-9 **and** rejects random scribbles as "Junk" (Class 10).